## Main processing of the data for exploring word embeddings in the context of Ukrainian-English translation

Libraries to install

In [ ]:
import fasttext
import numpy as np

from alignment import (
    learn_least_squares,
    build_candidate_matrices,
    translate_word
)

[FastText](https://fasttext.cc/docs/en/crawl-vectors.html) embedding models for Ukrainian and English

In [ ]:
model_uk = fasttext.load_model('../model/cc.uk.300.bin')
model_en = fasttext.load_model('../model/cc.en.300.bin')

Creating pairs of words - Ukrainian and English

In [ ]:
def data_to_dictionary(filename: str)-> dict:
    """
    Function creates from the data dictionary for 
    further usage

    :param filename: str, path to the data
    :return: dict, created dictionary
    """
    dict_res = {}
    with open(filename, "r", encoding="utf-8") as f_train:
        next(f_train)
        for line in f_train:
            line = line.strip("\n")
            ua_word, eng_word = line.split(" ")

            if ua_word not in dict_res:
                dict_res.setdefault(ua_word, [])

            dict_res[ua_word].append(eng_word)
    
    return dict_res

In [ ]:
train_dict = data_to_dictionary("../data/usage/uk-en-train.csv")
test_dict = data_to_dictionary("../data/usage/uk-en-test.csv")
full_dictionary = data_to_dictionary("../data/usage/uk-en-full.csv")

In [ ]:
pairs_train = []

for ua, eng_words in train_dict.items():
    for eng in eng_words:
        pairs_train.append((ua, eng))

Building aligned embeddings matrices for Ukrainian and English words

In [ ]:
ua_embeddings, eng_embeddings = [], []

for ua, eng in pairs_train:
    ua_embeddings.append(model_uk.get_word_vector(ua))
    eng_embeddings.append(model_en.get_word_vector(eng))

A_ua, B_eng = np.column_stack(ua_embeddings), np.column_stack(eng_embeddings)

Learning the least Square and Orthogonal Bilingual Mapping

In [ ]:
W_ls = learn_least_squares(A_ua, B_eng)
print("Least squares matrix of shape:", W_ls.shape)

Preparing English Candidate words for translation

In [ ]:
candidate_words = []
for _, eng_words in full_dictionary.items():
    for eng in eng_words:
        candidate_words.append(eng)

candidate_words, eng_candidate = build_candidate_matrices(candidate_words, model_en)
print("Number of candidate words:", len(candidate_words))
print("Candidate matrix shape:", eng_candidate .shape)

Simple translation of a Ukrainian word

In [ ]:
translate_word("кава", W_ls, model_uk, candidate_words, eng_candidate, top_k=5)